# Comment note-t-on un film ? Exploitation de la base de données MovieLens 32M

## Introduction et problématique

# I - MovieLens 32M : téléchargement et aperçu des données

In [1]:
import os
import zipfile

import requests
import pandas as pd

URL = "https://files.grouplens.org/datasets/movielens/ml-32m.zip"
NOM_ZIP = "ml-32m.zip"
DOSSIER = "data"

In [2]:
def telecharger(url, nom_fichier):
    """Télécharge le fichier à l'adresse url (par morceaux, car il est lourd)."""
    if os.path.exists(nom_fichier):
        print("Le ZIP existe déjà, pas de nouveau téléchargement.")
        return
    reponse = requests.get(url, stream=True)
    reponse.raise_for_status()
    with open(nom_fichier, "wb") as f:
        for morceau in reponse.iter_content(chunk_size=1024 * 1024):
            f.write(morceau)
    print("Téléchargement terminé.")


def extraire(nom_zip, dossier):
    """Dézippe l'archive dans le dossier indiqué."""
    with zipfile.ZipFile(nom_zip) as archive:
        print("Contenu du ZIP :", archive.namelist())
        archive.extractall(dossier)

In [3]:
telecharger(URL, NOM_ZIP)
extraire(NOM_ZIP, DOSSIER)

Téléchargement terminé.
Contenu du ZIP : ['ml-32m/', 'ml-32m/tags.csv', 'ml-32m/links.csv', 'ml-32m/README.txt', 'ml-32m/checksums.txt', 'ml-32m/ratings.csv', 'ml-32m/movies.csv']


In [4]:
chemin = os.path.join(DOSSIER, "ml-32m")

movies = pd.read_csv(os.path.join(chemin, "movies.csv"))
tags = pd.read_csv(os.path.join(chemin, "tags.csv"))
links = pd.read_csv(os.path.join(chemin, "links.csv"))
ratings = pd.read_csv(os.path.join(chemin, "ratings.csv"))

On affiche le premier dataframe, qui comporte une clé d'identification des films (movieId), le titre ainsi que leur année, et le genre associé. 
La base de données comporte 87585 films.

In [5]:
print(movies.shape)
movies.head()

(87585, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


On affiche aussi les notes de ces films, qui vont être notre base de donnée principale. Le fichier est particulièrement volumineux (32 millions de notes). L'userId permet de voir quel utilisateur a mis quelle note, ce qui sera utile. 

In [6]:
print(ratings.shape)
ratings.head()

(32000204, 4)


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


La base de données donne aussi accès à des tags, c'est-à-dire des mots que les utilisateurs ont associé aux films. Nous choisissons de ne pas les étudier dans le cadre de ce projet.

In [7]:
print(tags.shape)
tags.head()

(2000072, 4)


,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


Ce dataframe permet de faire des jointures avec des bases de données extérieures, en donnant la clé correspondante pour les films dans cette base et dans les bases IMDB et TMDB.

In [8]:
print(links.shape)
links.head()

(87585, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
